# Cross-Lingual Retrieval Evaluation

This notebook evaluates the retrieval performance of the multilingual semantic search system.

Research Question:

Can multilingual embeddings retrieve semantically equivalent documents across languages?

Evaluation Metrics:

- Recall@K
- Mean Reciprocal Rank (MRR)

The goal is to measure retrieval quality rather than simply inspect individual results.

Pipeline:
```
Cross-Lingual Query
|
↓
Semantic Search Engine
|
↓
Ranked Retrieval Results
|
↓
Evaluation Metrics
|
↓
Retrieval Performance Analysis
```

### Import Libraries and Project Components

In [1]:
import sys
from pathlib import Path

parent_dir = str(Path.cwd().parent)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from embeddings.encoder import MultilingualEncoder
from indexing.document_store import DocumentStore
from indexing.vector_index import VectorIndex
from retrieval.semantic_search import SemanticSearchEngine

import pandas as pd

### Initialize Semantic Search Pipeline

In [6]:
encoder = MultilingualEncoder() # Initialize the multilingual encoder
document_store = DocumentStore() # Initialize the document store

# Initialize the documents to be added to the document store
documents = [
    {
        "language": "English",
        "text": "The government introduced a new environmental policy.",
        "metadata": {
            "concept": "environment_policy"
        }
    },
    {
        "language": "German",
        "text": "Die Regierung führte eine neue Umweltpolitik ein.",
        "metadata": {
            "concept": "environment_policy"
        }
    },
    {
        "language": "Russian",
        "text": "Правительство ввело новую экологическую политику.",
        "metadata": {
            "concept": "environment_policy"
        }
    },
    {
        "language": "German",
        "text": "Das Unternehmen entwickelt erneuerbare Energietechnologien.",
        "metadata": {
            "concept": "renewable_energy"
        }
    },
    {
        "language": "Russian",
        "text": "Компания разрабатывает технологии возобновляемой энергии.",
        "metadata": {
            "concept": "renewable_energy"
        }
    }
]

# Iterate through the documents and add them to the document store
for document in documents:
    document_store.add_document(
        text=document["text"],
        language=document["language"],
        metadata=document["metadata"]
    )

# Encode the documents and add them to the vector index
document_texts = [document["text"] for document in documents]
# Encode the documents using the multilingual encoder
document_embeddings = encoder.encode(document_texts)
# Create a vector index and add the document embeddings to it
vector_index = VectorIndex(document_embeddings.shape[1])
# Add the document embeddings to the vector index
vector_index.add(document_embeddings)

# Initialize the semantic search engine with the encoder, vector index, 
# and document store
search_engine = SemanticSearchEngine(
    encoder=encoder,
    vector_index=vector_index,
    doc_store=document_store
)

print("Evaluation pipeline ready.")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluation pipeline ready.


### Create Evaluation Dataset

In [ ]:

# Define evaluation queries for testing the semantic search engine
evaluation_queries = [
    {
        "query": "The government created a new environmental policy.",
        "language": "English",
        "expected_concept": "environment_policy"
    },
    {
        "query": "Die Regierung führte eine neue Umweltpolitik ein.",
        "language": "German",
        "expected_concept": "environment_policy"
    },
    {
        "query": "Правительство ввело новую экологическую политику.",
        "language": "Russian",
        "expected_concept": "environment_policy"
    },
    {
        "query": "clean energy technology",
        "language": "English",
        "expected_concept": "renewable_energy"
    }
]

# Create a DataFrame to hold the evaluation queries and their expected concepts
evaluation_df = pd.DataFrame(evaluation_queries)
evaluation_df

,query,language,expected_concept
0,The government created a new environmental pol...,English,environment_policy
1,Die Regierung führte eine neue Umweltpolitik ein.,German,environment_policy
2,Правительство ввело новую экологическую политику.,Russian,environment_policy
3,clean energy technology,English,renewable_energy


### Evaluate Recall@K

In [7]:
def calculate_recall_at_k(search_engine, queries, k):
    """Calculate recall at k for a set of evaluation queries."""
    successful = 0

    # Iterate through each query in the evaluation set
    for item in queries:
        # Perform a search using the semantic search engine
        results = search_engine.search(
            item["query"],
            top_k=k
        )

        # Extract the concepts from the search results
        concepts = [
            result.document.metadata["concept"]
            for result in results
        ]

        # Check if the expected concept is present in the retrieved concepts
        if item["expected_concept"] in concepts:
            successful += 1

    # Calculate recall at k as the ratio of successful queries to total queries
    return successful / len(queries)


# Calculate recall at k for the evaluation queries
recall_5 = calculate_recall_at_k(
    search_engine,
    evaluation_queries,
    5
)

# Print the recall at k value
print(f"Recall@5: {recall_5:.2f}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Recall@5: 1.00


### Calculate Mean Reciprocal Rank (MRR)

In [8]:
def calculate_mrr(search_engine, queries, k):
    """Calculate Mean Reciprocal Rank (MRR) for a set of evaluation queries."""
    reciprocal_ranks = [] # Initialize list to hold reciprocal ranks for each query

    # Iterate through each query in the evaluation set
    for item in queries:
        # Perform a search using the semantic search engine
        results = search_engine.search(
            item["query"],
            top_k=k
        )

        # Extract the concepts from the search results
        rank_found = None

        # Iterate through search results to find rank of expected concept
        for rank, result in enumerate(results, start=1):
            # Check if the concept of current result matches expected concept
            if (
                result.document.metadata["concept"]
                == item["expected_concept"]
            ):
                rank_found = rank
                break

        # Calculate reciprocal rank for query and append to list
        if rank_found:
            reciprocal_ranks.append(
                1 / rank_found
            )
        # If the expected concept is not found in results, append 0 to list
        else:
            reciprocal_ranks.append(0)

    # Calculate and return the mean of the reciprocal ranks
    return sum(reciprocal_ranks) / len(reciprocal_ranks)


# Calculate MRR for the evaluation queries
mrr = calculate_mrr(
    search_engine,
    evaluation_queries,
    5
)

# Print the MRR value
print(f"MRR: {mrr:.2f}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

MRR: 1.00


### Retrieval Performance Summary

In [10]:
evaluation_results = pd.DataFrame(
    [
        {
            "Metric": "Recall@5",
            "Score": recall_5
        },
        {
            "Metric": "MRR",
            "Score": mrr
        }
    ]
)

evaluation_results

,Metric,Score
0,Recall@5,1.0
1,MRR,1.0
